# Kaggle image embedding model benchmark

Configure and export image embeddings using either Transformers-compatible CLIP/SigLIP models or Meta PE-Core models.

In [1]:
# CELL 01 — Configuration
from __future__ import annotations

import hashlib
import json
import os
import tempfile
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

INPUT_DIR = Path("/kaggle/input/datasets/trieu241007/keyframes-p1/keyframes")
OUTPUT_ROOT = Path("/kaggle/working/model_benchmark")
ARCHIVE_PATH = Path("/kaggle/working/model_benchmark.zip")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
RUNTIME_ROOT = OUTPUT_ROOT / "runtime"
METADATA_ROOT = OUTPUT_ROOT / "metadata"
OVERWRITE = False

PE_CORE_MODEL_PREFIX = "facebook/PE-Core-"
PE_REPOSITORY_URL = "https://github.com/facebookresearch/perception_models.git"
PE_REPOSITORY_REVISION = "3e352cca660658d4b5c90f42a7808b11469e4c66"
PE_SOURCE_ROOT = RUNTIME_ROOT / "perception_models"

MODELS = [
    {
        "name": "pe-core-l14-336",
        "model_id": "facebook/PE-Core-L14-336",
        "revision": None,
        "batch_size": 24,
        "inference_dtype": "float16",
    },
]

MODEL_NAME_PATTERN = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]*$")
RESERVED_MODEL_NAMES = {"runtime", "metadata", "checkpoints"}


def model_backend(model_id: str) -> str:
    return "pe_core" if model_id.startswith(PE_CORE_MODEL_PREFIX) else "transformers"


def pe_config_name(model_id: str) -> str:
    if model_backend(model_id) != "pe_core":
        raise ValueError(f"Not a PE Core model ID: {model_id}")
    name = model_id.rsplit("/", 1)[-1]
    if not name.startswith("PE-Core-"):
        raise ValueError(f"Invalid PE Core configuration name: {name}")
    return name


def pe_source_revision_for_model(model_id: str) -> str | None:
    return PE_REPOSITORY_REVISION if model_backend(model_id) == "pe_core" else None


def validate_models(models: list[dict[str, object]]) -> None:
    if not models:
        raise ValueError("MODELS must contain at least one model")
    names: set[str] = set()
    for model in models:
        name = model.get("name")
        model_id = model.get("model_id")
        revision = model.get("revision")
        batch_size = model.get("batch_size")
        inference_dtype = model.get("inference_dtype")
        if not isinstance(name, str) or MODEL_NAME_PATTERN.fullmatch(name) is None:
            raise ValueError(f"Unsafe model output name: {name!r}")
        normalized_name = name.casefold()
        if normalized_name in RESERVED_MODEL_NAMES:
            raise ValueError(f"Reserved model output name: {name}")
        if normalized_name in names:
            raise ValueError(f"Duplicate model output name: {name}")
        if not isinstance(model_id, str) or not model_id:
            raise ValueError(f"Invalid model_id for {name}")
        if model_backend(model_id) == "pe_core":
            pe_config_name(model_id)
        if revision is not None and (not isinstance(revision, str) or not revision):
            raise ValueError(f"Invalid revision for {name}")
        if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size <= 0:
            raise ValueError(f"Invalid batch_size for {name}: {batch_size!r}")
        if inference_dtype not in {"float16", "float32"}:
            raise ValueError(f"Invalid inference_dtype for {name}: {inference_dtype!r}")
        names.add(normalized_name)


def stable_hash(value: object) -> str:
    payload = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def atomic_write_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{path.name}-", suffix=".tmp", dir=path.parent
    )
    os.close(descriptor)
    temporary_path = Path(temporary_name)
    try:
        temporary_path.write_text(
            json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        temporary_path.replace(path)
    finally:
        temporary_path.unlink(missing_ok=True)


def output_facts(paths: list[Path]) -> list[dict[str, object]]:
    return [
        {
            "path": str(path),
            "kind": "directory" if path.is_dir() else "file",
            "size": path.stat().st_size if path.is_file() else None,
        }
        for path in paths
    ]


def checkpoint_is_valid(
    checkpoint_path: Path, fingerprint: str, outputs: list[Path]
) -> bool:
    if not checkpoint_path.is_file() or not all(path.exists() for path in outputs):
        return False
    try:
        checkpoint = json.loads(checkpoint_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return (
        checkpoint.get("schema_version") == "1.0"
        and checkpoint.get("fingerprint") == fingerprint
        and checkpoint.get("outputs") == output_facts(outputs)
    )


def complete_checkpoint(
    checkpoint_path: Path,
    cell_id: str,
    fingerprint: str,
    outputs: list[Path],
) -> None:
    atomic_write_json(
        checkpoint_path,
        {
            "schema_version": "1.0",
            "cell_id": cell_id,
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "fingerprint": fingerprint,
            "outputs": output_facts(outputs),
        },
    )


for directory in (OUTPUT_ROOT, CHECKPOINT_ROOT, RUNTIME_ROOT, METADATA_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

validate_models(MODELS)
configuration_fingerprint = stable_hash(
    {
        "input_dir": str(INPUT_DIR),
        "output_root": str(OUTPUT_ROOT),
        "models": MODELS,
        "pe_repository_url": PE_REPOSITORY_URL,
        "pe_repository_revision": PE_REPOSITORY_REVISION,
    }
)
complete_checkpoint(
    CHECKPOINT_ROOT / "cell_01_configuration.json",
    "cell-01-configuration",
    configuration_fingerprint,
    [CHECKPOINT_ROOT, RUNTIME_ROOT, METADATA_ROOT],
)
print("CELL 01 COMPLETE")

CELL 01 COMPLETE


In [2]:
# CELL 02 — Dependencies
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

from packaging.requirements import Requirement


def requirements_satisfied(required_packages: list[str]) -> bool:
    for package in required_packages:
        requirement = Requirement(package)
        try:
            installed_version = version(requirement.name)
        except PackageNotFoundError:
            return False
        if requirement.specifier and installed_version not in requirement.specifier:
            return False
    return True


def pe_source_revision(path: Path) -> str | None:
    try:
        completed = subprocess.run(
            ["git", "-C", str(path), "rev-parse", "HEAD"],
            check=True,
            capture_output=True,
            text=True,
        )
    except (OSError, subprocess.CalledProcessError):
        return None
    revision = completed.stdout.strip().lower()
    return revision if re.fullmatch(r"[0-9a-f]{40}", revision) else None


def pe_source_is_valid(path: Path) -> bool:
    required_files = (
        path / "core" / "vision_encoder" / "pe.py",
        path / "core" / "vision_encoder" / "config.py",
        path / "core" / "vision_encoder" / "transforms.py",
        path / "core" / "vision_encoder" / "bpe_simple_vocab_16e6.txt.gz",
    )
    return (
        all(candidate.is_file() for candidate in required_files)
        and pe_source_revision(path) == PE_REPOSITORY_REVISION
    )


def remove_runtime_child(path: Path) -> None:
    if path.resolve().parent != RUNTIME_ROOT.resolve():
        raise RuntimeError(f"Unsafe PE runtime path: {path}")
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def ensure_pe_source() -> None:
    if pe_source_is_valid(PE_SOURCE_ROOT):
        return
    clone_path = RUNTIME_ROOT / ".perception_models-clone"
    remove_runtime_child(PE_SOURCE_ROOT)
    remove_runtime_child(clone_path)
    try:
        subprocess.run(
            ["git", "init", str(clone_path)],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(clone_path), "remote", "add", "origin", PE_REPOSITORY_URL],
            check=True,
        )
        subprocess.run(
            [
                "git",
                "-C",
                str(clone_path),
                "fetch",
                "--depth",
                "1",
                "origin",
                PE_REPOSITORY_REVISION,
            ],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(clone_path), "checkout", "--detach", "FETCH_HEAD"],
            check=True,
        )
        if not pe_source_is_valid(clone_path):
            raise RuntimeError("Downloaded perception_models source is incomplete")
        clone_path.replace(PE_SOURCE_ROOT)
    finally:
        remove_runtime_child(clone_path)


def configure_pe_import_path() -> None:
    source_text = str(PE_SOURCE_ROOT)
    existing = [item for item in os.environ.get("PYTHONPATH", "").split(os.pathsep) if item]
    os.environ["PYTHONPATH"] = os.pathsep.join(dict.fromkeys([source_text, *existing]))
    if source_text not in sys.path:
        sys.path.insert(0, source_text)


requires_pe_core = any(
    model_backend(str(model["model_id"])) == "pe_core" for model in MODELS
)
required_packages = [
    "transformers>=4.49,<6",
    "Pillow>=10,<13",
    "huggingface_hub>=0.28,<2",
]
if requires_pe_core:
    required_packages.extend(
        ["einops==0.8.1", "timm==1.0.15", "ftfy>=6,<7", "regex>=2024"]
    )

dependency_outputs = [PE_SOURCE_ROOT] if requires_pe_core else []
dependency_fingerprint = stable_hash(
    {
        "requirements": required_packages,
        "python": sys.version,
        "requires_pe_core": requires_pe_core,
        "pe_repository_url": PE_REPOSITORY_URL,
        "pe_repository_revision": PE_REPOSITORY_REVISION,
    }
)
dependency_checkpoint = CHECKPOINT_ROOT / "cell_02_dependencies.json"

dependencies_ready = (
    checkpoint_is_valid(
        dependency_checkpoint, dependency_fingerprint, dependency_outputs
    )
    and requirements_satisfied(required_packages)
    and (not requires_pe_core or pe_source_is_valid(PE_SOURCE_ROOT))
)

if dependencies_ready:
    print("CELL 02 SKIPPED")
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *required_packages],
        check=True,
    )
    if not requirements_satisfied(required_packages):
        raise RuntimeError("Required dependency versions are unavailable after pip install")
    if requires_pe_core:
        ensure_pe_source()
    complete_checkpoint(
        dependency_checkpoint,
        "cell-02-dependencies",
        dependency_fingerprint,
        dependency_outputs,
    )
    print("CELL 02 COMPLETE")

import huggingface_hub
import transformers
from PIL import Image

if requires_pe_core:
    configure_pe_import_path()
    import core.vision_encoder.pe as pe
    import core.vision_encoder.transforms as pe_transforms


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>


Initialized empty Git repository in /kaggle/working/model_benchmark/runtime/.perception_models-clone/.git/


From https://github.com/facebookresearch/perception_models
 * branch            3e352cca660658d4b5c90f42a7808b11469e4c66 -> FETCH_HEAD
HEAD is now at 3e352cc Updated README for PE-AV to add new indicators.


CELL 02 COMPLETE


In [3]:
# CELL 03 — Environment
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable")
if torch.cuda.device_count() != 2:
    raise RuntimeError(
        f"Exactly two CUDA GPUs are required, found {torch.cuda.device_count()}"
    )

GPU_FACTS = []
for index in range(2):
    properties = torch.cuda.get_device_properties(index)
    GPU_FACTS.append(
        {
            "index": index,
            "name": properties.name,
            "capability": list(torch.cuda.get_device_capability(index)),
            "total_memory_bytes": properties.total_memory,
        }
    )

environment_fingerprint = stable_hash(
    {"configuration": configuration_fingerprint, "gpus": GPU_FACTS}
)
complete_checkpoint(
    CHECKPOINT_ROOT / "cell_03_environment.json",
    "cell-03-environment",
    environment_fingerprint,
    [],
)
for gpu in GPU_FACTS:
    print(gpu)
print("CELL 03 COMPLETE")

{'index': 0, 'name': 'Tesla T4', 'capability': [7, 5], 'total_memory_bytes': 15636037632}
{'index': 1, 'name': 'Tesla T4', 'capability': [7, 5], 'total_memory_bytes': 15636037632}
CELL 03 COMPLETE


In [4]:
# CELL 04 — Index
import re

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
KEYFRAME_PATTERN = re.compile(
    r"^(?P<video_id>.+)_s(?P<shot_id>\d+)_f(?P<frame_id>\d+)"
    r"(?P<extension>\.[^.]+)$",
    re.IGNORECASE,
)


def parse_keyframe_filename(filename: str) -> tuple[str, int, int]:
    match = KEYFRAME_PATTERN.fullmatch(filename)
    if match is None or match.group("extension").casefold() not in IMAGE_EXTENSIONS:
        raise ValueError(f"Malformed keyframe filename: {filename}")
    return (
        match.group("video_id"),
        int(match.group("shot_id")),
        int(match.group("frame_id")),
    )


def scan_images(
    input_dir: Path,
) -> tuple[list[dict[str, object]], list[tuple[str, int, int]]]:
    if not input_dir.is_dir():
        raise FileNotFoundError(f"Input image directory does not exist: {input_dir}")
    entries: list[tuple[str, Path, int, int]] = []
    with os.scandir(input_dir) as iterator:
        for entry in iterator:
            if entry.is_file(follow_symlinks=False) and Path(entry.name).suffix.casefold() in IMAGE_EXTENSIONS:
                stat = entry.stat(follow_symlinks=False)
                entries.append((entry.name, Path(entry.path).resolve(), stat.st_size, stat.st_mtime_ns))
    entries.sort(key=lambda item: (item[0].casefold(), item[0]))
    if not entries:
        raise ValueError(f"No supported images found in {input_dir}")
    folded: set[str] = set()
    indexed: list[dict[str, object]] = []
    fingerprint_facts: list[tuple[str, int, int]] = []
    for row, (filename, path, size, mtime_ns) in enumerate(entries):
        if filename.casefold() in folded:
            raise ValueError(f"Case-insensitive duplicate filename: {filename}")
        folded.add(filename.casefold())
        video_id, shot_id, frame_id = parse_keyframe_filename(filename)
        indexed.append(
            {
                "row": row,
                "path": str(path),
                "video_id": video_id,
                "shot_id": shot_id,
                "frame_id": frame_id,
            }
        )
        fingerprint_facts.append((filename, size, mtime_ns))
    return indexed, fingerprint_facts


def public_keyframe_record(item: dict[str, object]) -> dict[str, object]:
    return {
        "row": item["row"],
        "video_id": item["video_id"],
        "shot_id": item["shot_id"],
        "frame_id": item["frame_id"],
    }


JSONL_READ_CHUNK_BYTES = 64 * 1024


def compact_jsonl_line(record: dict[str, object]) -> bytes:
    return (
        json.dumps(
            record,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        + "\n"
    ).encode("utf-8")


def atomic_write_jsonl(path: Path, records: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{path.name}-", suffix=".tmp", dir=path.parent
    )
    temporary_path = Path(temporary_name)
    try:
        with os.fdopen(descriptor, "wb") as handle:
            for record in records:
                handle.write(compact_jsonl_line(record))
        temporary_path.replace(path)
    finally:
        temporary_path.unlink(missing_ok=True)


def jsonl_matches_records(path: Path, records: object) -> bool:
    try:
        with path.open("rb") as handle:
            for record in records:
                expected = compact_jsonl_line(record)
                for offset in range(0, len(expected), JSONL_READ_CHUNK_BYTES):
                    expected_chunk = expected[offset : offset + JSONL_READ_CHUNK_BYTES]
                    if handle.read(len(expected_chunk)) != expected_chunk:
                        return False
            return handle.read(1) == b""
    except OSError:
        return False


IMAGE_INDEX_PATH = METADATA_ROOT / "image_index.jsonl"
KEYFRAMES_PATH = METADATA_ROOT / "keyframes.jsonl"
indexed, fingerprint_facts = scan_images(INPUT_DIR)
IMAGE_COUNT = len(indexed)
index_fingerprint = stable_hash(
    {"configuration": configuration_fingerprint, "images": fingerprint_facts}
)
index_checkpoint = CHECKPOINT_ROOT / "cell_04_index.json"

if (
    checkpoint_is_valid(
        index_checkpoint, index_fingerprint, [IMAGE_INDEX_PATH, KEYFRAMES_PATH]
    )
    and jsonl_matches_records(IMAGE_INDEX_PATH, indexed)
    and jsonl_matches_records(
        KEYFRAMES_PATH,
        (public_keyframe_record(item) for item in indexed),
    )
):
    print("CELL 04 SKIPPED")
else:
    atomic_write_jsonl(IMAGE_INDEX_PATH, indexed)
    atomic_write_jsonl(
        KEYFRAMES_PATH, (public_keyframe_record(item) for item in indexed)
    )
    complete_checkpoint(
        index_checkpoint,
        "cell-04-index",
        index_fingerprint,
        [IMAGE_INDEX_PATH, KEYFRAMES_PATH],
    )
    print("CELL 04 COMPLETE")

CELL 04 COMPLETE


In [5]:
# CELL 05 — Runner
RUNNER_PATH = RUNTIME_ROOT / "two_gpu_embedding_runner.py"
RUNNER_SOURCE = r"""from __future__ import annotations

import argparse
import hashlib
import json
import os
import re
import shutil
import time
from contextlib import ExitStack
from pathlib import Path
from typing import Any

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import torch.multiprocessing as mp
from PIL import Image, ImageOps
from transformers import AutoModel, AutoProcessor
from transformers.utils import logging as transformers_logging

transformers_logging.disable_progress_bar()
transformers_logging.set_verbosity_error()

MODEL_NAME_PATTERN = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]*$")
RESERVED_MODEL_NAMES = {"runtime", "metadata", "checkpoints"}
PE_CORE_MODEL_PREFIX = "facebook/PE-Core-"


def model_backend(model_id: str) -> str:
    return "pe_core" if model_id.startswith(PE_CORE_MODEL_PREFIX) else "transformers"


def pe_config_name(model_id: str) -> str:
    if model_backend(model_id) != "pe_core":
        raise ValueError(f"Not a PE Core model ID: {model_id}")
    name = model_id.rsplit("/", 1)[-1]
    if not name.startswith("PE-Core-"):
        raise ValueError(f"Invalid PE Core configuration name: {name}")
    return name


def pe_checkpoint_path(snapshot_path: Path, model_id: str) -> Path:
    checkpoint_path = snapshot_path / f"{pe_config_name(model_id)}.pt"
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Missing PE Core checkpoint: {checkpoint_path}")
    return checkpoint_path


def validate_pe_source_revision(
    model_id: str, pe_source_revision: str | None
) -> str | None:
    if model_backend(model_id) != "pe_core":
        return None
    if not isinstance(pe_source_revision, str) or re.fullmatch(
        r"[0-9a-f]{40}", pe_source_revision
    ) is None:
        raise ValueError("A pinned 40-character PE source revision is required")
    return pe_source_revision


def validate_model_name(name: str) -> None:
    if MODEL_NAME_PATTERN.fullmatch(name) is None:
        raise ValueError(f"Unsafe model output name: {name!r}")
    if name.casefold() in RESERVED_MODEL_NAMES:
        raise ValueError(f"Reserved model output name: {name}")


def split_ranges(row_count: int) -> tuple[tuple[int, int], tuple[int, int]]:
    midpoint = (row_count + 1) // 2
    return (0, midpoint), (midpoint, row_count)


def stable_hash(value: object) -> str:
    payload = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def atomic_write_json(path: Path, value: object) -> None:
    temporary_path = path.with_name(f".{path.name}.{os.getpid()}.tmp")
    temporary_path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(path)


def processor_fingerprint(processor: object) -> str:
    to_dict = getattr(processor, "to_dict", None)
    configuration = to_dict() if callable(to_dict) else repr(processor)
    return stable_hash(
        {
            "class": f"{type(processor).__module__}.{type(processor).__qualname__}",
            "configuration": configuration,
        }
    )


def empty_worker_result(rank: int, start: int, end: int) -> dict[str, object]:
    return {
        "rank": rank,
        "start": start,
        "end": end,
        "dimension": None,
        "processor_fingerprint": None,
        "model_load_seconds": 0.0,
        "embedding_seconds": 0.0,
        "shard_path": None,
    }


def worker(
    rank: int,
    ranges: tuple[tuple[int, int], tuple[int, int]],
    index_path_text: str,
    output_root_text: str,
    name: str,
    model_id: str,
    revision: str | None,
    snapshot_path_text: str,
    batch_size: int,
    inference_dtype: str,
) -> None:
    start, end = ranges[rank]
    job_dir = Path(output_root_text) / "runtime" / f"{name}.job"
    if start == end:
        atomic_write_json(
            job_dir / f"worker-{rank}.json",
            empty_worker_result(rank, start, end),
        )
        print(f"GPU {rank}: empty range complete", flush=True)
        return
    try:
        device = torch.device(f"cuda:{rank}")
        torch.cuda.set_device(device)
        model_dtype = torch.float16 if inference_dtype == "float16" else torch.float32
        backend = model_backend(model_id)
        snapshot_path = Path(snapshot_path_text)
        load_started = time.perf_counter()
        if backend == "pe_core":
            try:
                import core.vision_encoder.pe as pe
                import core.vision_encoder.transforms as pe_transforms
            except ImportError as exc:
                raise RuntimeError(
                    "PE Core requires the official perception_models source from Cell 02"
                ) from exc
            config_name = pe_config_name(model_id)
            checkpoint_path = pe_checkpoint_path(snapshot_path, model_id)
            model = pe.CLIP.from_config(
                config_name,
                pretrained=True,
                checkpoint_path=str(checkpoint_path),
            ).eval().to(device)
            preprocess = pe_transforms.get_image_transform(model.image_size)
            preprocessing_fingerprint = stable_hash(
                {
                    "backend": backend,
                    "config_name": config_name,
                    "image_size": int(model.image_size),
                    "resize": "squash-bilinear",
                    "normalization": [[0.5, 0.5, 0.5], [0.5, 0.5, 0.5]],
                }
            )
        else:
            processor = AutoProcessor.from_pretrained(
                model_id, revision=revision, local_files_only=True
            )
            model = AutoModel.from_pretrained(
                model_id,
                revision=revision,
                local_files_only=True,
                torch_dtype=model_dtype,
            ).eval().to(device)
            if not callable(getattr(model, "get_image_features", None)):
                raise TypeError(f"Model does not expose get_image_features: {model_id}")
            preprocessing_fingerprint = processor_fingerprint(processor)
        model_load_seconds = time.perf_counter() - load_started
        print(f"GPU {rank}: model loaded", flush=True)

        assigned = read_jsonl(Path(index_path_text))[start:end]
        if len(assigned) != end - start:
            raise RuntimeError(f"Worker {rank} received an incomplete index range")
        shard_path = job_dir / f"worker-{rank}.npy"
        shard: np.memmap | None = None
        embedding_dimension: int | None = None
        embedding_started = time.perf_counter()
        total_batches = (len(assigned) + batch_size - 1) // batch_size

        for batch_number, offset in enumerate(range(0, len(assigned), batch_size), 1):
            batch = assigned[offset : offset + batch_size]
            with ExitStack() as stack:
                images = []
                for record in batch:
                    image = stack.enter_context(Image.open(record["path"]))
                    images.append(ImageOps.exif_transpose(image).convert("RGB"))
                if backend == "pe_core":
                    pixel_values = torch.stack(
                        [preprocess(image) for image in images]
                    ).to(device=device, non_blocking=True)
                    with torch.inference_mode(), torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                        enabled=inference_dtype == "float16",
                    ):
                        features = model.encode_image(pixel_values, normalize=False)
                else:
                    pixel_values = processor(images=images, return_tensors="pt")[
                        "pixel_values"
                    ].to(device=device, dtype=model_dtype, non_blocking=True)
                    with torch.inference_mode():
                        features = model.get_image_features(pixel_values=pixel_values)

            if hasattr(features, "pooler_output"):
                if features.pooler_output is None:
                    raise ValueError(
                        f"Model returned an empty pooler_output: {model_id}"
                    )
                features = features.pooler_output
            array = np.ascontiguousarray(
                features.detach().float().cpu().numpy(), dtype=np.float32
            )
            if array.ndim != 2 or array.shape[0] != len(batch):
                raise ValueError(
                    f"Unexpected feature shape for {model_id}: {array.shape}"
                )
            if not np.isfinite(array).all():
                raise ValueError(f"Non-finite features produced by {model_id}")
            norms = np.linalg.norm(array, axis=1, keepdims=True)
            if not np.isfinite(norms).all() or np.any(norms <= np.float32(1e-9)):
                raise ValueError(f"Invalid feature norms produced by {model_id}")
            array = np.ascontiguousarray(
                array / np.maximum(norms, np.float32(1e-9)), dtype=np.float32
            )

            if shard is None:
                embedding_dimension = int(array.shape[1])
                shard = np.lib.format.open_memmap(
                    shard_path,
                    mode="w+",
                    dtype=np.float32,
                    shape=(end - start, embedding_dimension),
                )
            elif array.shape[1] != embedding_dimension:
                raise ValueError(f"Embedding dimension changed within worker {rank}")
            shard[offset : offset + len(batch)] = array
            if batch_number % 25 == 0 or batch_number == total_batches:
                print(
                    f"GPU {rank}: completed batch {batch_number}/{total_batches}",
                    flush=True,
                )

        if shard is None or embedding_dimension is None:
            raise RuntimeError(f"Worker {rank} received an empty range")
        shard.flush()
        result = {
            "rank": rank,
            "start": start,
            "end": end,
            "dimension": embedding_dimension,
            "processor_fingerprint": preprocessing_fingerprint,
            "model_load_seconds": model_load_seconds,
            "embedding_seconds": time.perf_counter() - embedding_started,
            "shard_path": str(shard_path),
        }
        atomic_write_json(job_dir / f"worker-{rank}.json", result)
        print(f"GPU {rank}: complete", flush=True)
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError(
            f"CUDA OOM for model {model_id}, rank {rank}, batch size {batch_size}; "
            "lower this model's batch_size"
        ) from exc


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--index", type=Path, required=True)
    parser.add_argument("--metadata", type=Path, required=True)
    parser.add_argument("--index-fingerprint", required=True)
    parser.add_argument("--runner-sha256", required=True)
    parser.add_argument("--output-root", type=Path, required=True)
    parser.add_argument("--name", required=True)
    parser.add_argument("--model-id", required=True)
    parser.add_argument("--revision")
    parser.add_argument("--snapshot-path", type=Path, required=True)
    parser.add_argument("--pe-source-revision")
    parser.add_argument("--batch-size", type=int, required=True)
    parser.add_argument(
        "--inference-dtype", choices=("float16", "float32"), required=True
    )
    parser.add_argument("--overwrite", action="store_true")
    return parser.parse_args()


def checked_child(parent: Path, child: Path) -> Path:
    if child.resolve().parent != parent.resolve():
        raise RuntimeError(f"Unsafe output path: {child}")
    return child


def validate_worker_results(
    workers: list[dict[str, object]],
    ranges: tuple[tuple[int, int], tuple[int, int]],
) -> tuple[int, str]:
    observed_ranges = tuple(
        (int(result["start"]), int(result["end"])) for result in workers
    )
    if observed_ranges != ranges:
        raise RuntimeError(f"Unexpected worker ranges: {observed_ranges}")
    nonempty_results: list[dict[str, object]] = []
    for expected_rank, result in enumerate(workers):
        start, end = int(result["start"]), int(result["end"])
        if result.get("rank") != expected_rank:
            raise RuntimeError(f"Unexpected worker rank: {result}")
        if start == end:
            expected_empty = empty_worker_result(int(result["rank"]), start, end)
            if result != expected_empty:
                raise RuntimeError(f"Invalid empty worker result: {result}")
        else:
            if (
                not isinstance(result.get("dimension"), int)
                or int(result["dimension"]) <= 0
                or not isinstance(result.get("processor_fingerprint"), str)
                or not result["processor_fingerprint"]
                or not isinstance(result.get("shard_path"), str)
            ):
                raise RuntimeError(f"Invalid non-empty worker result: {result}")
            nonempty_results.append(result)
    if not nonempty_results:
        raise RuntimeError("No non-empty worker result")
    dimensions = {int(result["dimension"]) for result in nonempty_results}
    fingerprints = {
        str(result["processor_fingerprint"]) for result in nonempty_results
    }
    if len(dimensions) != 1:
        raise RuntimeError("Worker embedding dimensions do not match")
    if len(fingerprints) != 1:
        raise RuntimeError("Worker processor fingerprints do not match")
    return dimensions.pop(), fingerprints.pop()


def public_worker_result(result: dict[str, object]) -> dict[str, object]:
    dimension = result["dimension"]
    return {
        "rank": int(result["rank"]),
        "start": int(result["start"]),
        "end": int(result["end"]),
        "dimension": None if dimension is None else int(dimension),
        "model_load_seconds": float(result["model_load_seconds"]),
        "embedding_seconds": float(result["embedding_seconds"]),
    }


def main() -> None:
    args = parse_args()
    validate_model_name(args.name)
    if args.batch_size <= 0:
        raise ValueError("--batch-size must be positive")
    if not args.snapshot_path.is_dir():
        raise FileNotFoundError(f"Model snapshot does not exist: {args.snapshot_path}")
    pe_source_revision = validate_pe_source_revision(
        args.model_id, args.pe_source_revision
    )
    if model_backend(args.model_id) == "pe_core":
        pe_checkpoint_path(args.snapshot_path, args.model_id)
    if torch.cuda.device_count() != 2:
        raise RuntimeError(
            f"Exactly two visible CUDA GPUs are required, found {torch.cuda.device_count()}"
        )
    index_records = read_jsonl(args.index)
    row_count = len(index_records)
    if row_count < 1:
        raise ValueError("At least one indexed image is required")
    if [record.get("row") for record in index_records] != list(range(row_count)):
        raise ValueError("Image index rows must be contiguous and ordered")
    metadata_bytes = args.metadata.read_bytes()
    if metadata_bytes.count(b"\n") != row_count:
        raise ValueError("Metadata newline count does not match the image index")

    args.output_root.mkdir(parents=True, exist_ok=True)
    runtime_root = args.output_root / "runtime"
    runtime_root.mkdir(parents=True, exist_ok=True)
    final_dir = checked_child(args.output_root, args.output_root / args.name)
    publishing_dir = checked_child(
        args.output_root, args.output_root / f".{args.name}.publishing"
    )
    job_dir = checked_child(runtime_root, runtime_root / f"{args.name}.job")
    if final_dir.exists():
        if not args.overwrite:
            raise FileExistsError(
                f"Final model output already exists; pass --overwrite: {final_dir}"
            )
        shutil.rmtree(final_dir)
    for stale_path in (publishing_dir, job_dir):
        if stale_path.exists():
            if not args.overwrite:
                raise FileExistsError(
                    f"Stale model output exists; pass --overwrite: {stale_path}"
                )
            shutil.rmtree(stale_path)
    job_dir.mkdir(parents=False)

    ranges = split_ranges(row_count)
    started = time.perf_counter()
    mp.spawn(
        worker,
        args=(
            ranges,
            str(args.index),
            str(args.output_root),
            args.name,
            args.model_id,
            args.revision,
            str(args.snapshot_path),
            args.batch_size,
            args.inference_dtype,
        ),
        nprocs=2,
        join=True,
    )
    workers = [
        json.loads((job_dir / f"worker-{rank}.json").read_text(encoding="utf-8"))
        for rank in range(2)
    ]
    embedding_dimension, shared_processor_fingerprint = validate_worker_results(
        workers, ranges
    )

    publishing_dir.mkdir(parents=False)
    final = np.lib.format.open_memmap(
        publishing_dir / "embeddings.npy",
        mode="w+",
        dtype=np.float32,
        shape=(row_count, embedding_dimension),
    )
    for result in workers:
        start, end = int(result["start"]), int(result["end"])
        if start == end:
            continue
        shard_path = Path(result["shard_path"])
        shard = np.load(shard_path, mmap_mode="r", allow_pickle=False)
        if shard.shape != (end - start, embedding_dimension) or shard.dtype != np.float32:
            raise RuntimeError(f"Invalid worker shard: {shard_path}")
        final[start:end] = shard
    final.flush()

    elapsed_seconds = time.perf_counter() - started
    public_workers = [public_worker_result(result) for result in workers]
    benchmark = {
        "schema_version": "1.0",
        "model_id": args.model_id,
        "backend": model_backend(args.model_id),
        "revision": args.revision,
        "pe_source_revision": pe_source_revision,
        "processor_fingerprint": shared_processor_fingerprint,
        "index_fingerprint": args.index_fingerprint,
        "runner_sha256": args.runner_sha256,
        "image_count": row_count,
        "embedding_dimension": embedding_dimension,
        "batch_size_per_gpu": args.batch_size,
        "inference_dtype": args.inference_dtype,
        "elapsed_seconds": elapsed_seconds,
        "images_per_second": row_count / elapsed_seconds,
        "gpu_names": [torch.cuda.get_device_name(rank) for rank in range(2)],
        "workers": public_workers,
    }
    atomic_write_json(publishing_dir / "benchmark.json", benchmark)
    shutil.copyfile(args.metadata, publishing_dir / "keyframes.jsonl")
    matrix = np.load(
        publishing_dir / "embeddings.npy", mmap_mode="r", allow_pickle=False
    )
    if matrix.shape != (row_count, embedding_dimension) or matrix.dtype != np.float32:
        raise RuntimeError("Published embedding matrix header validation failed")
    if (publishing_dir / "keyframes.jsonl").read_bytes().count(b"\n") != row_count:
        raise RuntimeError("Published metadata newline validation failed")
    publishing_dir.rename(final_dir)
    shutil.rmtree(job_dir)


if __name__ == "__main__":
    main()
"""
runner_sha256 = hashlib.sha256(RUNNER_SOURCE.encode("utf-8")).hexdigest()
runner_checkpoint = CHECKPOINT_ROOT / "cell_05_runner.json"

if (
    checkpoint_is_valid(runner_checkpoint, runner_sha256, [RUNNER_PATH])
    and hashlib.sha256(RUNNER_PATH.read_bytes()).hexdigest() == runner_sha256
):
    print("CELL 05 SKIPPED")
else:
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{RUNNER_PATH.name}-", suffix=".tmp", dir=RUNNER_PATH.parent
    )
    temporary_path = Path(temporary_name)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="\n") as handle:
            handle.write(RUNNER_SOURCE)
        temporary_path.replace(RUNNER_PATH)
    finally:
        temporary_path.unlink(missing_ok=True)
    complete_checkpoint(
        runner_checkpoint, "cell-05-runner", runner_sha256, [RUNNER_PATH]
    )
    print("CELL 05 COMPLETE")

CELL 05 COMPLETE


In [6]:
# CELL 06 — Model cache
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()


def cached_snapshot_from_checkpoint(
    checkpoint_path: Path, fingerprint: str
) -> Path | None:
    if not checkpoint_path.is_file():
        return None
    try:
        checkpoint = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        outputs = checkpoint.get("outputs")
        if not isinstance(outputs, list) or len(outputs) != 1:
            return None
        snapshot_path = Path(outputs[0]["path"])
    except (KeyError, OSError, TypeError, json.JSONDecodeError):
        return None
    if checkpoint_is_valid(checkpoint_path, fingerprint, [snapshot_path]):
        return snapshot_path
    return None


def snapshot_is_valid(snapshot_path: Path, model_id: str) -> bool:
    if model_backend(model_id) == "pe_core":
        return (snapshot_path / f"{pe_config_name(model_id)}.pt").is_file()
    return (
        (snapshot_path / "config.json").is_file()
        and (snapshot_path / "preprocessor_config.json").is_file()
        and any(snapshot_path.glob("*.safetensors"))
    )


MODEL_SNAPSHOT_PATHS: dict[str, Path] = {}
all_models_cached = True
for model in MODELS:
    model_name = str(model["name"])
    model_id = str(model["model_id"])
    revision = model.get("revision")
    cache_fingerprint = stable_hash(
        {
            "model_id": model_id,
            "revision": revision,
            "backend": model_backend(model_id),
        }
    )
    cache_checkpoint = CHECKPOINT_ROOT / f"cell_06_model_cache__{model_name}.json"
    snapshot_path = cached_snapshot_from_checkpoint(
        cache_checkpoint, cache_fingerprint
    )
    if snapshot_path is None or not snapshot_is_valid(snapshot_path, model_id):
        all_models_cached = False
        snapshot_path = Path(snapshot_download(repo_id=model_id, revision=revision))
        if not snapshot_is_valid(snapshot_path, model_id):
            raise RuntimeError(f"Incomplete model snapshot for {model_id}: {snapshot_path}")
        complete_checkpoint(
            cache_checkpoint,
            "cell-06-model-cache",
            cache_fingerprint,
            [snapshot_path],
        )
    MODEL_SNAPSHOT_PATHS[model_name] = snapshot_path
    print(f"Model cache ready: {model_name}")

print("CELL 06 SKIPPED" if all_models_cached else "CELL 06 COMPLETE")

Model cache ready: pe-core-l14-336
CELL 06 COMPLETE


In [7]:
# CELL 07 — Embeddings
import shutil
import numpy as np

PUBLIC_MODEL_ARTIFACTS = {"embeddings.npy", "keyframes.jsonl", "benchmark.json"}


def model_output_is_valid(model_dir: Path, model: dict[str, object]) -> bool:
    embeddings_path = model_dir / "embeddings.npy"
    keyframes_path = model_dir / "keyframes.jsonl"
    benchmark_path = model_dir / "benchmark.json"
    try:
        if {path.name for path in model_dir.iterdir()} != PUBLIC_MODEL_ARTIFACTS:
            return False
    except OSError:
        return False
    if not all(path.is_file() for path in (embeddings_path, keyframes_path, benchmark_path)):
        return False
    try:
        matrix = np.load(embeddings_path, mmap_mode="r", allow_pickle=False)
        benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
        keyframes_bytes = keyframes_path.read_bytes()
        expected_keyframes_bytes = KEYFRAMES_PATH.read_bytes()
    except (OSError, ValueError, TypeError, json.JSONDecodeError):
        return False
    return (
        matrix.ndim == 2
        and matrix.shape[0] == IMAGE_COUNT
        and matrix.shape[1] > 0
        and matrix.dtype == np.float32
        and keyframes_bytes == expected_keyframes_bytes
        and benchmark.get("schema_version") == "1.0"
        and benchmark.get("index_fingerprint") == index_fingerprint
        and benchmark.get("runner_sha256") == runner_sha256
        and benchmark.get("model_id") == model["model_id"]
        and benchmark.get("backend") == model_backend(str(model["model_id"]))
        and benchmark.get("revision") == model.get("revision")
        and benchmark.get("pe_source_revision") == pe_source_revision_for_model(str(model["model_id"]))
        and benchmark.get("batch_size_per_gpu") == model["batch_size"]
        and benchmark.get("inference_dtype") == model["inference_dtype"]
        and benchmark.get("image_count") == IMAGE_COUNT
        and benchmark.get("embedding_dimension") == matrix.shape[1]
    )


all_models_skipped = True
for model in MODELS:
    model_name = str(model["name"])
    model_dir = OUTPUT_ROOT / model_name
    embedding_fingerprint = stable_hash(
        {"index": index_fingerprint, "model": model, "runner": runner_sha256}
    )
    embedding_checkpoint = (
        CHECKPOINT_ROOT / f"cell_07_embeddings__{model_name}.json"
    )
    public_outputs = [
        model_dir / "embeddings.npy",
        model_dir / "keyframes.jsonl",
        model_dir / "benchmark.json",
    ]

    if not OVERWRITE and model_output_is_valid(model_dir, model):
        if not checkpoint_is_valid(
            embedding_checkpoint, embedding_fingerprint, public_outputs
        ):
            complete_checkpoint(
                embedding_checkpoint,
                "cell-07-embeddings",
                embedding_fingerprint,
                public_outputs,
            )
        print(f"Model embeddings ready: {model_name} (skipped)")
        continue

    all_models_skipped = False
    if model_dir.resolve().parent != OUTPUT_ROOT.resolve():
        raise RuntimeError(f"Unsafe model output path: {model_dir}")
    if model_dir.exists():
        shutil.rmtree(model_dir)

    publishing_dir = OUTPUT_ROOT / f".{model_name}.publishing"
    if publishing_dir.resolve().parent != OUTPUT_ROOT.resolve():
        raise RuntimeError(f"Unsafe publishing path: {publishing_dir}")
    if publishing_dir.exists():
        shutil.rmtree(publishing_dir)

    job_dir = RUNTIME_ROOT / f"{model_name}.job"
    if job_dir.resolve().parent != RUNTIME_ROOT.resolve():
        raise RuntimeError(f"Unsafe runtime job path: {job_dir}")
    if job_dir.exists():
        shutil.rmtree(job_dir)

    command = [
        sys.executable,
        "-u",
        str(RUNNER_PATH),
        "--index", str(IMAGE_INDEX_PATH),
        "--metadata", str(KEYFRAMES_PATH),
        "--index-fingerprint", index_fingerprint,
        "--runner-sha256", runner_sha256,
        "--output-root", str(OUTPUT_ROOT),
        "--name", model["name"],
        "--model-id", model["model_id"],
        "--snapshot-path", str(MODEL_SNAPSHOT_PATHS[model_name]),
        "--batch-size", str(model["batch_size"]),
        "--inference-dtype", model["inference_dtype"],
    ]
    if model.get("revision") is not None:
        command.extend(["--revision", str(model["revision"])])
    pe_revision = pe_source_revision_for_model(str(model["model_id"]))
    if pe_revision is not None:
        command.extend(["--pe-source-revision", pe_revision])
    if OVERWRITE:
        command.append("--overwrite")
    subprocess.run(
        command, check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"}
    )
    if not model_output_is_valid(model_dir, model):
        raise RuntimeError(f"Published model output failed validation: {model_name}")
    complete_checkpoint(
        embedding_checkpoint,
        "cell-07-embeddings",
        embedding_fingerprint,
        public_outputs,
    )
    print(f"Model embeddings ready: {model_name}")

print("CELL 07 SKIPPED" if all_models_skipped else "CELL 07 COMPLETE")

Missing keys for loading model: []
Unexpected keys for loading model: []
Missing keys for loading model: []
Unexpected keys for loading model: []
GPU 0: model loaded
GPU 1: model loaded
GPU 1: completed batch 25/8646
GPU 0: completed batch 25/8646
GPU 1: completed batch 50/8646
GPU 0: completed batch 50/8646
GPU 1: completed batch 75/8646
GPU 0: completed batch 75/8646
GPU 1: completed batch 100/8646
GPU 0: completed batch 100/8646
GPU 1: completed batch 125/8646
GPU 0: completed batch 125/8646
GPU 1: completed batch 150/8646
GPU 0: completed batch 150/8646
GPU 1: completed batch 175/8646
GPU 0: completed batch 175/8646
GPU 1: completed batch 200/8646
GPU 0: completed batch 200/8646
GPU 1: completed batch 225/8646
GPU 0: completed batch 225/8646
GPU 1: completed batch 250/8646
GPU 0: completed batch 250/8646
GPU 1: completed batch 275/8646
GPU 0: completed batch 275/8646
GPU 1: completed batch 300/8646
GPU 0: completed batch 300/8646
GPU 1: completed batch 325/8646
GPU 0: completed bat

In [8]:
# CELL 08 — Validation
VALIDATION_CHUNK_ROWS = 8192
BINARY_CHUNK_BYTES = 8 * 1024 * 1024
PUBLIC_MODEL_ARTIFACTS = {"embeddings.npy", "keyframes.jsonl", "benchmark.json"}


def binary_metadata_is_valid(path: Path) -> bool:
    newline_count = 0
    try:
        with path.open("rb") as candidate, KEYFRAMES_PATH.open("rb") as expected:
            while True:
                candidate_chunk = candidate.read(BINARY_CHUNK_BYTES)
                expected_chunk = expected.read(BINARY_CHUNK_BYTES)
                if candidate_chunk != expected_chunk:
                    return False
                newline_count += candidate_chunk.count(b"\n")
                if not candidate_chunk:
                    return newline_count == IMAGE_COUNT
    except OSError:
        return False


def validate_public_model_output(model_dir: Path, model: dict[str, object]) -> None:
    embeddings_path = model_dir / "embeddings.npy"
    metadata_path = model_dir / "keyframes.jsonl"
    benchmark_path = model_dir / "benchmark.json"
    try:
        direct_children = {path.name for path in model_dir.iterdir()}
    except OSError as exc:
        raise ValueError(f"Cannot inspect model artifacts for {model['name']}") from exc
    if direct_children != PUBLIC_MODEL_ARTIFACTS or not all(
        path.is_file() for path in (embeddings_path, metadata_path, benchmark_path)
    ):
        raise ValueError(
            f"Unexpected model artifacts for {model['name']}: {sorted(direct_children)}"
        )
    matrix = np.load(embeddings_path, mmap_mode="r", allow_pickle=False)
    try:
        if matrix.ndim != 2 or matrix.shape[0] != IMAGE_COUNT or matrix.dtype != np.float32:
            raise ValueError(
                f"Invalid embedding header for {model['name']}: {matrix.shape}, {matrix.dtype}"
            )
        embedding_dimension = int(matrix.shape[1])
        for start in range(0, IMAGE_COUNT, VALIDATION_CHUNK_ROWS):
            end = min(start + VALIDATION_CHUNK_ROWS, IMAGE_COUNT)
            block = np.asarray(matrix[start:end], dtype=np.float32)
            if not np.isfinite(block).all():
                raise ValueError(f"Non-finite vectors for {model['name']} at rows {start}:{end}")
            norms = np.linalg.norm(block, axis=1)
            if not np.all(np.abs(norms - np.float32(1.0)) <= np.float32(1e-5)):
                raise ValueError(f"Non-unit vectors for {model['name']} at rows {start}:{end}")
    finally:
        mmap_handle = getattr(matrix, "_mmap", None)
        if mmap_handle is not None:
            mmap_handle.close()

    if not binary_metadata_is_valid(metadata_path):
        raise ValueError(f"Invalid keyframe metadata for {model['name']}")
    try:
        benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        raise ValueError(f"Invalid benchmark metadata for {model['name']}") from exc
    expected_benchmark = {
        "schema_version": "1.0",
        "index_fingerprint": index_fingerprint,
        "runner_sha256": runner_sha256,
        "model_id": model["model_id"],
        "backend": model_backend(str(model["model_id"])),
        "revision": model.get("revision"),
        "pe_source_revision": pe_source_revision_for_model(str(model["model_id"])),
        "batch_size_per_gpu": model["batch_size"],
        "inference_dtype": model["inference_dtype"],
        "image_count": IMAGE_COUNT,
        "embedding_dimension": embedding_dimension,
    }
    if any(benchmark.get(key) != value for key, value in expected_benchmark.items()):
        raise ValueError(f"Benchmark provenance mismatch for {model['name']}")


def validation_checkpoint_is_current(
    checkpoint_path: Path,
    fingerprint: str,
    public_outputs: list[Path],
    model_dir: Path,
    model: dict[str, object],
) -> bool:
    validate_public_model_output(model_dir, model)
    return checkpoint_is_valid(checkpoint_path, fingerprint, public_outputs)


all_models_skipped = True
for model in MODELS:
    model_name = str(model["name"])
    model_dir = OUTPUT_ROOT / model_name
    public_outputs = [
        model_dir / "embeddings.npy",
        model_dir / "keyframes.jsonl",
        model_dir / "benchmark.json",
    ]
    embedding_fingerprint = stable_hash(
        {"index": index_fingerprint, "model": model, "runner": runner_sha256}
    )
    embedding_checkpoint = CHECKPOINT_ROOT / f"cell_07_embeddings__{model_name}.json"
    if not checkpoint_is_valid(embedding_checkpoint, embedding_fingerprint, public_outputs):
        raise RuntimeError(f"Missing or invalid Cell 07 checkpoint for {model_name}")
    validation_fingerprint = stable_hash(
        {"embedding": embedding_fingerprint, "chunk_rows": VALIDATION_CHUNK_ROWS}
    )
    validation_checkpoint = CHECKPOINT_ROOT / f"cell_08_validation__{model_name}.json"
    if validation_checkpoint_is_current(
        validation_checkpoint,
        validation_fingerprint,
        public_outputs,
        model_dir,
        model,
    ):
        print(f"Model validation ready: {model_name} (skipped)")
        continue
    all_models_skipped = False
    complete_checkpoint(
        validation_checkpoint, "cell-08-validation", validation_fingerprint, public_outputs
    )
    print(f"Model validation complete: {model_name}")

print("CELL 08 SKIPPED" if all_models_skipped else "CELL 08 COMPLETE")

Model validation complete: pe-core-l14-336
CELL 08 COMPLETE


In [9]:
# CELL 09 — Archive
import zipfile

PUBLIC_FILES = ("embeddings.npy", "keyframes.jsonl", "benchmark.json")


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(BINARY_CHUNK_BYTES), b""):
            digest.update(chunk)
    return digest.hexdigest()


def archive_checkpoint_is_valid(
    checkpoint_path: Path, source_fingerprint: str, archive_path: Path
) -> bool:
    if not checkpoint_path.is_file() or not archive_path.is_file():
        return False
    try:
        checkpoint = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        archive_sha256 = file_sha256(archive_path)
    except (OSError, json.JSONDecodeError):
        return False
    return (
        checkpoint.get("schema_version") == "1.0"
        and checkpoint.get("cell_id") == "cell-09-archive"
        and checkpoint.get("fingerprint") == source_fingerprint
        and checkpoint.get("outputs") == output_facts([archive_path])
        and checkpoint.get("archive_sha256") == archive_sha256
    )


def complete_archive_checkpoint(
    checkpoint_path: Path, source_fingerprint: str, archive_path: Path
) -> None:
    atomic_write_json(
        checkpoint_path,
        {
            "schema_version": "1.0",
            "cell_id": "cell-09-archive",
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "fingerprint": source_fingerprint,
            "outputs": output_facts([archive_path]),
            "archive_sha256": file_sha256(archive_path),
        },
    )


def write_public_archive(path: Path) -> None:
    with zipfile.ZipFile(
        path, mode="w", compression=zipfile.ZIP_STORED, allowZip64=True
    ) as archive:
        for model in MODELS:
            model_dir = OUTPUT_ROOT / str(model["name"])
            for filename in PUBLIC_FILES:
                archive.write(
                    model_dir / filename,
                    arcname=f"{model['name']}/{filename}",
                )


validated_artifacts: list[dict[str, object]] = []
for model in MODELS:
    model_name = str(model["name"])
    model_dir = OUTPUT_ROOT / model_name
    public_outputs = [model_dir / filename for filename in PUBLIC_FILES]
    embedding_fingerprint = stable_hash(
        {"index": index_fingerprint, "model": model, "runner": runner_sha256}
    )
    validation_fingerprint = stable_hash(
        {"embedding": embedding_fingerprint, "chunk_rows": VALIDATION_CHUNK_ROWS}
    )
    validation_checkpoint = CHECKPOINT_ROOT / f"cell_08_validation__{model_name}.json"
    if not checkpoint_is_valid(validation_checkpoint, validation_fingerprint, public_outputs):
        raise RuntimeError(f"Missing or invalid Cell 08 checkpoint for {model_name}")
    validate_public_model_output(model_dir, model)
    validated_artifacts.append(
        {
            "name": model_name,
            "files": [
                {"name": filename, "sha256": file_sha256(model_dir / filename)}
                for filename in PUBLIC_FILES
            ],
        }
    )

archive_fingerprint = stable_hash({"models": validated_artifacts})
archive_checkpoint = CHECKPOINT_ROOT / "cell_09_archive.json"
if archive_checkpoint_is_valid(archive_checkpoint, archive_fingerprint, ARCHIVE_PATH):
    print("CELL 09 SKIPPED")
else:
    ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{ARCHIVE_PATH.name}-", suffix=".tmp", dir=ARCHIVE_PATH.parent
    )
    os.close(descriptor)
    temporary_path = Path(temporary_name)
    try:
        write_public_archive(temporary_path)
        temporary_path.replace(ARCHIVE_PATH)
    finally:
        temporary_path.unlink(missing_ok=True)
    complete_archive_checkpoint(archive_checkpoint, archive_fingerprint, ARCHIVE_PATH)
    print("CELL 09 COMPLETE")

print(f"Download: {ARCHIVE_PATH}")

CELL 09 COMPLETE
Download: /kaggle/working/model_benchmark.zip
